# Notebook 01 — Reflexion: a self-improving loop

**Workshop:** Agentic AI for Scientists — Week 3 (From Loops to Graphs)
**Block:** Reflexion (22 min)
**Goal:** Build the **generate → reflect → revise** cycle as a LangGraph graph, and watch it measurably improve a **paper abstract** over a few turns. This is the research-writing loop, automated: draft, critique against criteria, rewrite to address the critique, repeat until good enough.

**Reflection vs Reflexion.** *Reflection* = a generator and a critic looping until "good enough". *Reflexion* (Shinn et al., 2023) adds the key idea that the **critique is persisted into state** and each revision must *address prior weaknesses* — so the agent doesn't just re-roll, it converges. We build exactly that: the critique lives in the graph state and feeds the next draft.


## 0. Setup

In [ ]:
%pip install -q \
    "langgraph>=1.2.4" "langchain>=1.3.2" "langchain-core>=1.4.0" \
    "langchain-google-genai>=4.2.4" python-dotenv

In [ ]:
import os

# Free Gemini API key (no credit card): https://aistudio.google.com/apikey
# Colab: add GOOGLE_API_KEY under the key icon (left sidebar) -> "Secrets".
# Local: put GOOGLE_API_KEY=AIza... in a .env file next to this notebook.
try:
    from google.colab import userdata  # type: ignore
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

# Accept GEMINI_API_KEY as an alias (some AI Studio users store it under that name).
if not os.environ.get("GOOGLE_API_KEY") and os.environ.get("GEMINI_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY first (see the comment above)."
print("API key loaded.")


## 1. State — carry the draft AND the running critique

The Reflexion idea is in the state shape: alongside the `draft`, we keep the latest `critique` and an `iteration` counter. The critique is what makes the loop *converge* instead of wander.


In [ ]:
from typing import TypedDict


class ReflexState(TypedDict):
    topic: str          # what the abstract is about (fixed input)
    draft: str          # the current abstract
    critique: str       # the latest critique (empty on round 1)
    score: int          # the critic's 1-10 quality score
    iteration: int      # how many revise rounds we've done


## 2. The critic returns *structured* output

We want a number we can branch on, not prose. `llm.with_structured_output(PydanticModel)` makes Gemini fill a schema. **Keep the model flat** — Gemini's JSON mode is happiest with simple fields (nested/Union types can fail).


In [ ]:
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)


class Critique(BaseModel):
    score: int = Field(description="Overall quality of the abstract, 1 (poor) to 10 (publishable).")
    suggestions: list[str] = Field(description="2-4 concrete, specific improvements to make next.")


critic = llm.with_structured_output(Critique)

# quick check
demo = critic.invoke("Critique this abstract: 'We did some experiments on cells and got results.'")
print("score:", demo.score)
print("suggestions:", demo.suggestions)


## 3. The three nodes

- **generate** — write (or rewrite) the abstract. On a revise round it's *given the prior critique* and told to address it.
- **reflect** — run the structured critic; store score + suggestions in state.
- The router (**should_continue**) ends when the score clears a bar or we hit max iterations; otherwise it loops back to `generate`.


In [ ]:
MAX_ITERS = 3
TARGET_SCORE = 8


def generate(state: ReflexState) -> dict:
    if state["critique"]:
        prompt = (
            f"Rewrite this abstract on '{state['topic']}' to address the critique.\n\n"
            f"CURRENT ABSTRACT:\n{state['draft']}\n\n"
            f"CRITIQUE TO ADDRESS:\n{state['critique']}\n\n"
            "Return ONLY the improved abstract (120-180 words)."
        )
    else:
        prompt = (
            f"Write a 120-180 word scientific abstract on: {state['topic']}. "
            "Include background, method, key result, and significance. Return ONLY the abstract."
        )
    return {"draft": llm.invoke(prompt).content}


def reflect(state: ReflexState) -> dict:
    c = critic.invoke(
        "You are a tough journal reviewer. Critique this abstract for clarity, specificity, "
        f"and whether it states a concrete result.\n\nABSTRACT:\n{state['draft']}"
    )
    return {
        "critique": "; ".join(c.suggestions),
        "score": c.score,
        "iteration": state["iteration"] + 1,
    }


def should_continue(state: ReflexState) -> str:
    if state["score"] >= TARGET_SCORE or state["iteration"] >= MAX_ITERS:
        return "stop"
    return "revise"


## 4. Wire the graph — `generate → reflect → (revise ↺ | stop)`

`generate` and `reflect` are nodes; the conditional edge after `reflect` is the Reflexion loop.


In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ReflexState)
builder.add_node("generate", generate)
builder.add_node("reflect", reflect)

builder.add_edge(START, "generate")
builder.add_edge("generate", "reflect")
builder.add_conditional_edges(
    "reflect",
    should_continue,
    {"revise": "generate",   # loop back with the critique in state
     "stop": END},
)

app = builder.compile()


**See the graph.** The compiled app can draw its own topology — this picture *is* the generate → reflect → revise loop you just wired.

In [ ]:
# --- See the graph this code builds: Reflexion loop ---
# A compiled LangGraph app can draw its own topology: nodes are steps,
# edges are control flow. This picture IS the agent you just wired.
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # draw_mermaid_png needs network/graphviz; ASCII always works
    print(app.get_graph().draw_ascii())

## 5. Run it — watch the score climb

`.stream()` prints state after each node, so you see draft → critique → revised draft, with the score rising toward the bar.


In [ ]:
init = {
    "topic": "a machine-learning method to predict sepsis onset from ICU vital-sign time series",
    "draft": "", "critique": "", "score": 0, "iteration": 0,
}

for step in app.stream(init):
    node, payload = next(iter(step.items()))
    if node == "reflect":
        print(f"\n[reflect] iteration {payload['iteration']}  score={payload['score']}")
        print("  suggestions:", payload["critique"])
    elif node == "generate":
        print(f"\n[generate] new draft:\n{payload['draft'][:400]}...")

final = app.invoke(init)
print("\n\n=== FINAL ABSTRACT (score %d, %d iterations) ===" % (final["score"], final["iteration"]))
print(final["draft"])


## 6. See the reasoning in LangSmith

Each node, each LLM call, the score at every turn — as a nested trace. On the LangGraph **1.x** line this works with **env vars alone** (no client wrapping): every call goes through a LangChain Runnable, so the tracer attaches automatically.

> Contrast with Week 2: there we pinned classic 0.3.x, whose tracer silently posted nothing on Colab and we had to wrap the raw client. On 1.x, just set the (newer) `LANGSMITH_*` env names below.


In [ ]:
# Optional: turn on LangSmith tracing. Safe with no key — tracing just stays off.
import os

def _secret(name):
    try:
        from google.colab import userdata  # type: ignore
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name)

ls_key = _secret("LANGSMITH_API_KEY") or _secret("LANGCHAIN_API_KEY")
if ls_key:
    os.environ["LANGSMITH_TRACING"] = "true"           # 1.x reads LANGSMITH_* (not LANGCHAIN_* like 0.3.x)
    os.environ["LANGSMITH_API_KEY"] = ls_key
    os.environ["LANGSMITH_PROJECT"] = _secret("LANGSMITH_PROJECT") or "agentic-ai-week3"
    print("LangSmith tracing ON  ->  project '" + os.environ["LANGSMITH_PROJECT"] + "'")
    print("Run the graph below, then open https://smith.langchain.com to watch each node.")
else:
    print("No LANGSMITH_API_KEY found -> tracing OFF (everything still runs).")
    print("Add it as a Colab Secret (key icon) to see every node/edge as a nested trace.")


In [ ]:
# Re-run with tracing on -> the whole loop appears as one trace tree in LangSmith.
final = app.invoke(init)
print("Done. Open https://smith.langchain.com -> project 'agentic-ai-week3' for the trace.")
print("Final score:", final["score"], "| iterations:", final["iteration"])


---

## Recap

- **Reflexion = generate → reflect → revise**, with the critique **persisted in state** so each revision is targeted, not random.
- The **structured critic** (`with_structured_output`) gives a number the conditional edge can branch on.
- The loop is a **cycle in the graph**, capped by score *or* iteration — exactly the control NB00 set up.

Next: **Agentic RAG** — retrieval that grades its own results and web-searches when the corpus comes up short.
